# 02 - RAG Generation with HF Wikipedia

This notebook runs headline-only RAG using the embedded Wikipedia dataset from Hugging Face (`not-lain/wikipedia`, revision `embedded`). The Colab workflow uses the complete generate-many/select-best pipeline: five retrieved contexts, five sampled candidates, structural validation, semantic LLM-Judge reranking, and greedy fallback. Word-pair examples are generated without retrieved context.

**Nota sul kernel**

Seleziona il kernel `LLM-Project (conda)` nel kernel picker in alto a destra prima di eseguire le celle. (Esempio: apri il kernel picker → scegli "LLM-Project (conda)")


## Shared settings

The complete matrix contains 15 runs: three models multiplied by five Wikipedia corpus sizes. The Colab cells explicitly use `k=5` and tag their outputs with `_bestof5_llmjudge`. The function defaults remain compatible with the existing local section.

In [3]:
from pathlib import Path
import os
import subprocess
import sys

MODELS = ("llama", "qwen", "mistral")
N_DOC_COUNTS = tuple(range(5_000, 25_001, 5_000))
DATA = "data/raw/task-a-en.tsv"
RAG_CONFIG = "configs/rag.yaml"
K = 4
APPLY_TO = "headline"
SKIP_EXISTING = False


def _project_root() -> Path:
    root = Path.cwd().resolve()
    if (root / "scripts" / "run_rag_generate.py").exists():
        return root
    candidate = root.parent
    if (candidate / "scripts" / "run_rag_generate.py").exists():
        return candidate
    raise RuntimeError(f"Could not locate the repository root from {root}")


def _use_mock_run(mock: bool | None = None) -> bool:
    if mock is not None:
        return mock
    try:
        import torch
    except ImportError:
        return True
    return not torch.cuda.is_available()


def run_rag_matrix(
    output_suffix="",
    mock: bool | None = None,
    k: int = K,
    strategy_suffix: str = "",
):
    root = _project_root()
    use_mock = _use_mock_run(mock)
    total = len(MODELS) * len(N_DOC_COUNTS)
    completed = 0

    for model in MODELS:
        for n_docs in N_DOC_COUNTS:
            if model == "llama" and not os.getenv("HF_TOKEN"):
                completed += 1
                print(f"[{completed}/{total}] Skipping model={model}, documents={n_docs:,} (HF_TOKEN missing).")
                continue

            output_path = root / Path(
                f"data/generated/rag/{model}_rag_wiki_headline_"
                f"{n_docs // 1000}k_k{k}{strategy_suffix}{output_suffix}.jsonl"
            )
            if SKIP_EXISTING and output_path.exists():
                completed += 1
                print(f"[{completed}/{total}] Skipping existing output: {output_path}")
                continue

            output_path.parent.mkdir(parents=True, exist_ok=True)
            cmd = [
                sys.executable, str(root / "scripts" / "run_rag_generate.py"),
                "--model", model,
                "--input", str(root / DATA),
                "--output", str(output_path),
                "--rag-config", str(root / RAG_CONFIG),
                "--n-docs", str(n_docs),
                "--k", str(k),
                "--apply-to", APPLY_TO,
                "--overwrite",
            ]
            if use_mock:
                cmd.append("--mock")
            print(
                f"[{completed + 1}/{total}] model={model}, documents={n_docs:,}, "
                f"k={k}, selection=best-of-5/LLM-Judge, mode={'mock' if use_mock else 'real'}"
            )
            subprocess.run(cmd, check=True, cwd=root)
            completed += 1
            print(f"Saved: {output_path}")

    print(f"Matrix complete: {completed}/{total} outputs available.")

print(f"Models: {', '.join(MODELS)}")
print(f"Wikipedia corpus sizes: {[f'{n // 1000}k' for n in N_DOC_COUNTS]}")
print(f"Runs per environment: {len(MODELS) * len(N_DOC_COUNTS)}")

Models: llama, qwen, mistral
Wikipedia corpus sizes: ['5k', '10k', '15k', '20k', '25k']
Runs per environment: 15


# Remote execution on Colab

Run this chapter in Google Colab. The repository is expected at `/content/LLM-Project-SemEval-Humor-Generation`. Llama also requires an accepted Hugging Face license and an authenticated Hugging Face account.

In [4]:
%cd /content/LLM-Project-SemEval-Humor-Generation
%pip install -r requirements-colab.txt

import os
from getpass import getpass

if not (os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")):
    token = getpass("Hugging Face token for Llama (hidden; leave blank to skip Llama): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
    else:
        print("HF_TOKEN not set: Llama runs will be skipped; Qwen and Mistral remain available.")

/content/LLM-Project-SemEval-Humor-Generation


## Run the complete Colab matrix

Set `RUN_COLAB_MATRIX` to `True` to launch all 15 combinations with `k=5`, five candidates, validation, semantic LLM-Judge reranking, and greedy fallback.

In [ ]:
RUN_COLAB_MATRIX = False
COLAB_K = 5
COLAB_STRATEGY_SUFFIX = "_bestof5_llmjudge"

if RUN_COLAB_MATRIX:
    run_rag_matrix(k=COLAB_K, strategy_suffix=COLAB_STRATEGY_SUFFIX)
else:
    print("Set RUN_COLAB_MATRIX = True to launch all 15 Colab runs.")

## Check Colab outputs

In [ ]:
for model in MODELS:
    for n_docs in N_DOC_COUNTS:
        output_path = Path(
            f"data/generated/rag/{model}_rag_wiki_headline_"
            f"{n_docs // 1000}k_k{COLAB_K}{COLAB_STRATEGY_SUFFIX}.jsonl"
        )
        status = "ready" if output_path.exists() else "missing"
        print(f"{status:7}  {output_path}")

## Optional diversity diagnostic (Colab)

This optional diagnostic measures surface diversity only; it is separate from the semantic LLM-Judge selection used by the complete pipeline. It now uses the same `k=5` retrieval setting.

The extra word pair is a test-only constraint layered onto the repository's headline prompt. The percentage is descriptive and is not used to select production outputs.

In [ ]:
from copy import deepcopy
from pathlib import Path
import difflib
import itertools
import os
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

# Test configuration: edit these values if needed.
TEST_MODEL = "llama"
TEST_N_DOCS = 5_000
TEST_K = 5
NUM_VARIANTS = 5
TEMPERATURE = 0.8
TOP_P = 0.9
MAX_NEW_TOKENS = 64
test_headline = "Autonomous car safely navigates city streets for the first time"
test_word_pair = "car, bumper"

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before running this test.")
if NUM_VARIANTS < 2:
    raise ValueError("NUM_VARIANTS must be at least 2.")

repo_root = Path.cwd().resolve()
if not (repo_root / "scripts" / "run_rag_generate.py").exists():
    raise RuntimeError("Run the Colab %cd setup cell first.")
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Ask securely when the token is not already available. getpass also works when
# the runtime is controlled from Jupyter/VS Code, where Colab Secrets time out.
if TEST_MODEL == "llama" and not (os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")):
    from getpass import getpass

    hf_token = getpass("Hugging Face token (input hidden): ").strip()
    if not hf_token:
        raise RuntimeError("A Hugging Face token is required for the configured Llama model.")
    os.environ["HF_TOKEN"] = hf_token

from humor_gen.models import get_runner
from humor_gen.prompts import build_generation_prompt
from humor_gen.rag import build_retriever
from humor_gen.utils import load_yaml, resolve_model_config
from humor_gen.validate import clean_joke

rag_cfg = load_yaml(repo_root / "configs/rag.yaml")
rag_cfg = deepcopy(rag_cfg)
rag_cfg.setdefault("retriever", {})["n_docs"] = TEST_N_DOCS
# Keep the embedding model on CPU so the generator has the Colab GPU memory.
rag_cfg["retriever"]["query_device"] = "cpu"

generation_cfg = load_yaml(repo_root / "configs/generation.yaml")
generation_cfg = deepcopy(generation_cfg)
generation_cfg.setdefault("generation", {}).update({
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "max_new_tokens": MAX_NEW_TOKENS,
})
model_cfg = resolve_model_config(TEST_MODEL, repo_root / "configs/models.yaml")

print(f"🔍 1. INTERROGAZIONE RAG (Wikipedia={TEST_N_DOCS:,}, k={TEST_K})...")
retriever = build_retriever(
    str(repo_root / "data/raw/task-a-en.tsv"),
    rag_cfg,
    mock=False,
)
query = f"Background facts and context about: {test_headline}"
retrieved_docs = retriever.retrieve(query, k=TEST_K)
if not retrieved_docs:
    raise RuntimeError("The retriever returned no Wikipedia contexts.")
for index, document in enumerate(retrieved_docs, start=1):
    preview = " ".join(document.split())[:180]
    print(f"  [{index}] {preview}{'…' if len(document) > 180 else ''}")

item = {
    "id": "colab-diversity-test",
    "input_type": "headline",
    "headline": test_headline,
}
word1, word2 = (part.strip() for part in test_word_pair.split(",", maxsplit=1))
prompt = build_generation_prompt(item, retrieved_docs)
if not prompt.endswith("Punchline:"):
    raise RuntimeError("Unexpected generation prompt format.")
prompt = (
    prompt[:-len("Punchline:")]
    + f'Additional constraint: naturally include both words "{word1}" and "{word2}".\n'
    + "Punchline:"
)

print(f"\n🤖 Caricamento di {model_cfg['display_name']}...")
runner = get_runner(model_cfg, generation_cfg, mock=False)

print(f"🎲 2. GENERAZIONE DI {NUM_VARIANTS} VARIANTI (temperature={TEMPERATURE})...")
variants = []
for index in range(NUM_VARIANTS):
    # The lower-level call is intentional: it lets this combined headline + word-pair test reuse the real prompt.
    text = clean_joke(runner._generate_text(prompt))
    variants.append(text)
    print(f'  ✨ Variante {index + 1}: "{text}"')

print("\n📊 3. ANALISI DELLA DIVERSITÀ")
diversity_matrix = np.zeros((NUM_VARIANTS, NUM_VARIANTS), dtype=float)
pairwise_diversities = []
for idx1, idx2 in itertools.combinations(range(NUM_VARIANTS), 2):
    similarity = difflib.SequenceMatcher(None, variants[idx1], variants[idx2]).ratio()
    diversity = 100.0 * (1.0 - similarity)
    diversity_matrix[idx1, idx2] = diversity
    diversity_matrix[idx2, idx1] = diversity
    pairwise_diversities.append(diversity)

labels = [f"V{i}" for i in range(1, NUM_VARIANTS + 1)]
display(pd.DataFrame(diversity_matrix, index=labels, columns=labels).round(1))
avg_diversity = float(np.mean(pairwise_diversities))
print(f"📈 DIVERSITÀ MEDIA DEL POOL CON k={TEST_K}: {avg_diversity:.2f}%")

if avg_diversity > 40:
    print(f"✅ Le cinque uscite mostrano una buona diversità di sequenza con k={TEST_K}.")
else:
    print("⚠️ Le uscite sono simili: prova prima più campionamento e poi confronta k diversi con lo stesso protocollo.")

## End-to-end generate-many + LLM-Judge smoke test (Colab)

Run this before the full matrix. It processes one real dataset record with Wikipedia `k=5`, generates five candidates at temperature 0.8, validates them, invokes the configured semantic reranker when at least two survive, and uses greedy fallback when none survive. The table exposes every intermediate decision.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

RUN_FULL_PIPELINE_SMOKE = False
SMOKE_MODEL = "llama"
SMOKE_N_DOCS = 25_000
SMOKE_K = 5
SMOKE_LIMIT = 1

root = _project_root()
smoke_output = root / (
    f"data/generated/rag/{SMOKE_MODEL}_rag_wiki_headline_"
    f"{SMOKE_N_DOCS // 1000}k_k{SMOKE_K}_bestof5_llmjudge_smoke.jsonl"
)

if RUN_FULL_PIPELINE_SMOKE:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Select a Colab GPU runtime before launching the real smoke test.")
    if SMOKE_MODEL == "llama" and not (os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")):
        raise RuntimeError("Set HF_TOKEN in the Colab setup cell, or select qwen/mistral.")

    command = [
        sys.executable,
        str(root / "scripts/run_rag_generate.py"),
        "--model", SMOKE_MODEL,
        "--input", str(root / DATA),
        "--output", str(smoke_output),
        "--rag-config", str(root / RAG_CONFIG),
        "--n-docs", str(SMOKE_N_DOCS),
        "--k", str(SMOKE_K),
        "--apply-to", "headline",
        "--limit", str(SMOKE_LIMIT),
        "--overwrite",
    ]
    print("Running the complete RAG → 5 candidates → validation → LLM-Judge pipeline...")
    subprocess.run(command, check=True, cwd=root)
else:
    print("Set RUN_FULL_PIPELINE_SMOKE = True to execute one real end-to-end record.")

if smoke_output.exists():
    with smoke_output.open(encoding="utf-8") as file:
        row = json.loads(next(line for line in file if line.strip()))

    metadata = row.get("metadata", {})
    reranker_info = metadata.get("reranker", {})
    candidate_rows = []
    for index, candidate in enumerate(metadata.get("candidates", [])):
        candidate_rows.append({
            "index": index,
            "valid": candidate.get("valid"),
            "reward": candidate.get("score"),
            "fallback": candidate.get("fallback"),
            "errors": ", ".join(candidate.get("constraint_errors", [])),
            "joke": candidate.get("generated_joke"),
        })

    display(pd.DataFrame(candidate_rows))
    print(f"Input: {row['input']}")
    print(f"Winner: {row['generated_joke']}")
    print(f"Final valid: {row['valid']}")
    print(f"Reranker: {reranker_info.get('scorer_type')}")
    print(f"Judge reason: {reranker_info.get('reasoning')}")
    print(f"Fallback used: {metadata.get('fallback_used')}")
    print(f"Saved to: {smoke_output}")
elif RUN_FULL_PIPELINE_SMOKE:
    raise RuntimeError(f"Expected output was not created: {smoke_output}")

# Local execution with the .venv kernel

Run this chapter on a local workstation after selecting the project `.venv` as the Jupyter kernel. The subprocesses use `sys.executable`, so generation runs with that same environment.

In [21]:
def _project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scripts" / "run_rag_generate.py").exists():
            return candidate
    raise RuntimeError(f"Could not locate the repository root from {current}")


ROOT = _project_root()
print(f"Repository root: {ROOT}")
print(f"Python interpreter: {sys.executable}")

Repository root: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation
Python interpreter: c:\Users\peppe\miniconda3\envs\llmproj\python.exe


## Local dependency install

Run this once in the selected `.venv` kernel. It can be skipped when the environment already contains the project dependencies and GPU stack.

In [22]:
%pip install -r requirements-colab.txt

import getpass
import os

os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ")
print("Token configurato per questa sessione.")

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-colab.txt'


Token configurato per questa sessione.


## Run the complete local matrix

Set `RUN_LOCAL_MATRIX` to `True` to launch all 15 combinations. Local output names use `_local` to avoid overwriting Colab results.

In [23]:
RUN_LOCAL_MATRIX = True

if RUN_LOCAL_MATRIX:
    run_rag_matrix(output_suffix="_local")
else:
    print("Set RUN_LOCAL_MATRIX = True to launch all 15 local runs.")

[1/15] model=llama, documents=5,000, mode=real
Saved: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_5k_k4_local.jsonl
[2/15] model=llama, documents=10,000, mode=real
Saved: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_10k_k4_local.jsonl
[3/15] model=llama, documents=15,000, mode=real
Saved: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_15k_k4_local.jsonl
[4/15] model=llama, documents=20,000, mode=real
Saved: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_20k_k4_local.jsonl
[5/15] model=llama, documents=25,000, mode=real
Saved: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_25k_k4_local.jsonl
[6/15] model=qwen, documents=5,000, mode=real
Saved: C:\Users\peppe\OneD

## Check local outputs

In [24]:
for model in MODELS:
    for n_docs in N_DOC_COUNTS:
        # Usa ROOT per comporre il path assoluto esatto
        output_path = ROOT / Path(
            f"data/generated/rag/{model}_rag_wiki_headline_{n_docs // 1000}k_k{K}_local.jsonl"
        )
        status = "ready" if output_path.exists() else "missing"
        print(f"{status:7}  {output_path}")

ready    C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_5k_k4_local.jsonl
ready    C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_10k_k4_local.jsonl
ready    C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_15k_k4_local.jsonl
ready    C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_20k_k4_local.jsonl
ready    C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\llama_rag_wiki_headline_25k_k4_local.jsonl
ready    C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\qwen_rag_wiki_headline_5k_k4_local.jsonl
ready    C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\data\generated\rag\qwen_rag_wiki_headline_10k_k4_local.jsonl
ready    C:\Users\peppe\

In [25]:
import pandas as pd
import json

# Scegli il percorso del file jsonl che ha performato meglio (es. llama con 25k doc)
best_jsonl_path = ROOT / "data/generated/rag/llama_rag_wiki_headline_25k_k4_local_FIXED.jsonl"
output_tsv_path = ROOT / "outputs/submissions/subtask_a_en_submission.tsv"

# Assicurati che la cartella di output esista
output_tsv_path.parent.mkdir(parents=True, exist_ok=True)

# Leggi il JSONL riga per riga
data = []
with open(best_jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        data.append(json.loads(line))

# Crea il DataFrame generato
df_generated = pd.DataFrame(data)

# Carica il file di input originale per mantenere la struttura intatta (ID, ecc.)
df_original = pd.read_csv(ROOT / "data/raw/task-a-en.tsv", sep='\t')

# Unisci l'output generato basandoti sull'ID per non fare errori di allineamento (Left Join)
df_submission = df_original.merge(df_generated[['id', 'generated_joke']], on='id', how='left')

# Fallback sicuro per evetuali valori mancanti (NaN)
fallback_joke = "I ran out of time to invent a clever punchline."
df_submission['generated_joke'] = df_submission['generated_joke'].fillna(fallback_joke)

# Isola solo le colonne richieste dal formato ufficiale di sottomissione
df_submission = df_submission[['id', 'generated_joke']]

# Salva il file definitivo in TSV pronto per la sottomissione ufficiale, senza indice
df_submission.to_csv(output_tsv_path, sep='\t', index=False, encoding='utf-8')
print(f"File di sottomissione pronto in: {output_tsv_path}")

File di sottomissione pronto in: C:\Users\peppe\OneDrive\Desktop\LLM-Project-SemEval-Humor-Generation\outputs\submissions\subtask_a_en_submission.tsv


In [7]:
import sys
import torch
print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device_count:", torch.cuda.device_count())
    print("device_name:", torch.cuda.get_device_name(0))


python: c:\Users\peppe\miniconda3\envs\llmproj\python.exe
torch: 2.5.1
cuda_available: True
device_count: 1
device_name: NVIDIA GeForce RTX 4070


In [26]:
from __future__ import annotations
import re

# Abbiamo esteso il vocabolario delle stop-words per intercettare meglio le battute brevi in inglese
ENGLISH_HINT_WORDS = {
    "the", "a", "an", "and", "or", "to", "of", "in", "on", "with", 
    "for", "is", "are", "was", "were", "why", "when", "because", 
    "it", "its", "you", "that", "this", "they", "their", "them", "but", "not"
}

def clean_joke(text: str) -> str:
    text = (text or "").strip()
    # Rimuove in modo sicuro le premesse introdotte dagli LLM
    text = re.sub(r"^(sure[,.!]?\s*)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^(here('s| is)\s+a\s+joke[:\s-]*)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^(joke|answer|response)\s*:\s*", "", text, flags=re.IGNORECASE)
    return " ".join(text.strip().strip('"').split())

def validate_joke(joke: str, item: dict[str, str], max_words: int = 45) -> tuple[bool, list[str]]:
    errors: list[str] = []
    
    # Pulizia preventiva del testo prima di qualsiasi validazione
    cleaned_text = clean_joke(joke)
    lowered = cleaned_text.casefold()
    
    if not cleaned_text:
        errors.append("empty_output")
        return False, errors
        
    if len(cleaned_text.split()) > max_words:
        errors.append("too_long")
        
    if "\n" in cleaned_text:
        errors.append("multiple_lines")
        
    if any(marker in lowered for marker in ("this joke", "the humor", "explanation:", "the punchline is", "funny because")):
        errors.append("contains_explanation")
        
    if not _looks_english(cleaned_text):
        errors.append("not_english_like")
        
    # Controllo Vincolo: Coppia di Parole Rare
    if item["input_type"] == "word_inclusion":
        if not _contains_word(lowered, item["word1"]):
            errors.append("missing_word1")
        if not _contains_word(lowered, item["word2"]):
            errors.append("missing_word2")
            
    # Controllo Vincolo: Titolo di Giornale (Headline)
    if item["input_type"] == "headline":
        if not _has_headline_overlap(lowered, item.get("headline", "")):
            errors.append("low_headline_relevance")
            
    return not errors, errors

def _contains_word(lowered: str, word: str) -> bool:
    word = str(word).strip().casefold()
    if not word:
        return False
    # Supporta lemmi e plurali semplici terminanti in 's', 'es', 'ed' o 'ing' per evitare falsi blocchi
    # Esempio: se cerca 'weigh', accetta anche 'weights' o 'weighed' o 'weighing'
    pattern = rf"(?<![a-z0-9]){re.escape(word)}(s|es|ed|ing)?(?![a-z0-9])"
    return re.search(pattern, lowered) is not None or word in lowered

def _looks_english(text: str) -> bool:
    tokens = re.findall(r"[A-Za-z']+", text.casefold())
    if not tokens:
        return False
    ascii_ratio = sum(ch.isascii() for ch in text) / max(len(text), 1)
    hint_ratio = sum(tok in ENGLISH_HINT_WORDS for tok in tokens) / max(len(tokens), 1)
    
    # Ottimizzazione cruciale: allentata la soglia minima di stop-words al 4% 
    # e alzata la lunghezza a 12 token per salvare le battute fulminee
    return ascii_ratio > 0.9 and (hint_ratio > 0.04 or len(tokens) <= 12)

def _has_headline_overlap(lowered: str, headline: str) -> bool:
    # Estrae i token significativi riducendo la lunghezza minima a 3 caratteri (es. "US", "Open", "bus")
    headline_terms = {
        token
        for token in re.findall(r"[A-Za-z]{3,}", headline.casefold())
        if token not in ENGLISH_HINT_WORDS
    }
    if not headline_terms:
        return True
    
    # Controllo esteso: accetta la sovrapposizione testuale diretta
    return any(term in lowered for term in headline_terms)

In [ ]:
import json
from pathlib import Path

# Definiamo i percorsi dei due file da confrontare (usando la ROOT del progetto)
file_orig_path = ROOT / "data/generated/rag/llama_rag_wiki_headline_25k_k4_local.jsonl"
file_fixed_path = ROOT / "data/generated/rag/llama_rag_wiki_headline_25k_k4_local_FIXED.jsonl"

# Dizionari per mappare i record tramite ID
records_orig = {}
records_fixed = {}

# Carichiamo il file originale
with open(file_orig_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rec = json.loads(line)
            records_orig[rec["id"]] = rec

# Carichiamo il file FIXED
with open(file_fixed_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rec = json.loads(line)
            records_fixed[rec["id"]] = rec

# Liste per l'analisi
salvati = []
testi_cambiati = 0

for item_id, rec_fixed in records_fixed.items():
    rec_orig = records_orig.get(item_id)
    if not rec_orig:
        continue
        
    # 1. Verifichiamo se il testo della barzelletta è cambiato
    if rec_orig["generated_joke"] != rec_fixed["generated_joke"]:
        testi_cambiati += 1
        
    # 2. Identifichiamo i record "Salvati" (Invalido prima -> Valido ora)
    if not rec_orig["valid"] and rec_fixed["valid"]:
        salvati.append({
            "id": item_id,
            "input": rec_fixed["input"],
            "joke": rec_fixed["generated_joke"],
            "vecchi_errori": rec_orig["constraint_errors"]
        })

# --- STAMPA DEL REPORT ---
print("=" * 80)
print("📊 REPORT DI CONFRONTO: ORIGINALE vs FIXED (Llama 25k)")
print("=" * 80)
print(f"🔹 Testi delle barzellette modificati: {testi_cambiati} (Dovrebbe essere 0)")
print(f"🎉 Barzellette totali 'salvate' dai nuovi criteri: {len(salvati)}")
print("-" * 80)

if salvati:
    print("\n📝 ESEMPI DI BATTUTE SALVATE DALLA RI-VALIDAZIONE:\n")
    # Mostriamo i primi 3 casi eclatanti per l'analisi visiva
    for i, item in enumerate(salvati[:3], start=1):
        print(f"Es. {i} [ID: {item['id']}]")
        print(f"👉 Input:  {item['input']}")
        print(f"💬 Joke:   \"{item['joke']}\"")
        print(f"❌ Vecchi Errori che la penalizzavano: {item['vecchi_errori']}")
        print("-" * 50)
else:
    print("Nessun record è passato da Invalido a Valido.")

🚀 Inizio ri-validazione globale della matrice RAG...
----------------------------------------------------------------------

🎉 Classifica Matrice RAG Aggiornata:
Modello Wikipedia Corpus  Nuova Validità (%)                                                        Errori di Vincolo Rimanenti                                   Nuovo File
mistral              15k               86.58             {'low_headline_relevance': 151, 'not_english_like': 10, 'too_long': 1} mistral_rag_wiki_headline_15k_k4_local.jsonl
mistral              25k               85.83                             {'not_english_like': 8, 'low_headline_relevance': 162} mistral_rag_wiki_headline_25k_k4_local.jsonl
mistral              10k               85.50                            {'low_headline_relevance': 157, 'not_english_like': 17} mistral_rag_wiki_headline_10k_k4_local.jsonl
mistral               5k               85.17                            {'low_headline_relevance': 166, 'not_english_like': 13}  mistral_rag_wiki_